# Crop Yield Prediction — AIC354 Assignment 3

**Course:** Machine Learning Fundamentals (AIC354)  
**Problem type:** Regression  
**Target:** `hg/ha_yield` (crop yield in hectograms per hectare)  
**Evaluation:** 10-Fold Cross-Validation using RMSE  
**Dataset:** [Crop Yield Prediction Dataset on Kaggle](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset)

> Important: Run every cell in order. The numerical results are generated from the dataset you upload. Do not copy example/fabricated scores into your report.

## 1. Assignment plan

This notebook compares five regression algorithms that are different from the five in the provided car-price notebook:

1. Extra Trees Regressor
2. AdaBoost Regressor
3. Elastic Net
4. Histogram-based Gradient Boosting Regressor
5. SGD Regressor

The preprocessing steps are kept inside each model pipeline so that imputation, scaling, and categorical encoding are learned from each training fold only. This helps prevent data leakage.

## 2. Upload the Kaggle dataset

1. Download the dataset from the Kaggle link above.
2. Extract the downloaded ZIP file on your computer.
3. Find the combined file named `yield_df.csv` if it is included.
4. In the next cell, upload `yield_df.csv` (recommended) or the downloaded ZIP file.

The notebook uses the **full available dataset**, not a small sample.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path("/content/crop_yield_data")
DATA_DIR.mkdir(exist_ok=True)

for uploaded_name in uploaded.keys():
    source_path = Path("/content") / uploaded_name
    if uploaded_name.lower().endswith(".zip"):
        with zipfile.ZipFile(source_path, "r") as zf:
            zf.extractall(DATA_DIR)
    elif uploaded_name.lower().endswith(".csv"):
        shutil.copy2(source_path, DATA_DIR / uploaded_name)

csv_files = list(DATA_DIR.rglob("*.csv"))
print("CSV files found:")
for p in csv_files:
    print("-", p)

## 3. Load and inspect the full dataset

The combined Kaggle file is usually named `yield_df.csv`. If multiple CSVs are found, the notebook prefers the combined file and then checks for the expected target and feature columns.

In [ ]:
import pandas as pd
import numpy as np

if not csv_files:
    raise FileNotFoundError("No CSV file found. Upload yield_df.csv or the Kaggle ZIP file.")

preferred = [p for p in csv_files if "yield_df" in p.name.lower()]
if preferred:
    data_path = preferred[0]
else:
    # Choose the CSV most likely to already contain both the yield target and input features.
    candidates = []
    for p in csv_files:
        try:
            head = pd.read_csv(p, nrows=5)
            cols = {str(c).strip().lower() for c in head.columns}
            score = sum(x in cols for x in ["area", "item", "year", "hg/ha_yield", "average_rain_fall_mm_per_year", "avg_temp"])
            candidates.append((score, p))
        except Exception:
            pass
    if not candidates:
        raise ValueError("Could not read any CSV files. Please upload the combined yield_df.csv file.")
    data_path = sorted(candidates, key=lambda x: x[0], reverse=True)[0][1]

df = pd.read_csv(data_path)
df.columns = [str(c).strip() for c in df.columns]
print("Selected file:", data_path.name)
print("Full dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

## 4. Clean the data and select X and y

This version expects the combined dataset to contain these columns:

- `Area` — country/area
- `Item` — crop name
- `Year` — year of observation
- `average_rain_fall_mm_per_year` — annual rainfall
- `pesticides_tonnes` — pesticide usage
- `avg_temp` — average temperature
- `hg/ha_yield` — crop yield target

If your CSV uses different column names, inspect the printed column list and adjust the mapping below. The `Unnamed: 0` column, if present, is only an index and is removed.

In [ ]:
# Remove accidental index columns, if present.
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")].copy()

required = [
    "Area", "Item", "Year",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes", "avg_temp", "hg/ha_yield"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing expected columns: {missing}\n"
        f"Columns found: {df.columns.tolist()}\n"
        "Please use the combined yield_df.csv from the recommended Kaggle dataset."
    )

df = df[required].copy()
df = df.drop_duplicates()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
for col in ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "hg/ha_yield"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["hg/ha_yield", "Area", "Item", "Year"])
df["Area"] = df["Area"].astype(str)
df["Item"] = df["Item"].astype(str)

print("Rows after cleaning:", len(df))
print("Columns after cleaning:", df.columns.tolist())
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing values"))
display(df.describe(include="all").transpose())

## 5. Define input features and target

The model predicts the crop yield value stored in `hg/ha_yield`. This target is in hectograms per hectare. For a simple interpretation, divide the predicted value by 10,000 to express it as tonnes per hectare.

In [ ]:
FEATURES = [
    "Area", "Item", "Year",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes", "avg_temp"
]
TARGET = "hg/ha_yield"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Number of rows used:", len(X))
print("Number of input features:", len(FEATURES))
print("Target:", TARGET)
print("First 5 target values:")
display(y.head())

## 6. Build preprocessing and five alternative regression models

Numeric columns are imputed with the median and scaled. Text columns are imputed with the most frequent value and one-hot encoded. The preprocessing is part of each pipeline, so each cross-validation fold learns its preprocessing from its training portion only.

In [ ]:
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet, SGDRegressor
from sklearn.base import clone
import time
import matplotlib.pyplot as plt
import joblib
import json

print("scikit-learn version:", sklearn.__version__)

numeric_features = [
    "Year", "average_rain_fall_mm_per_year",
    "pesticides_tonnes", "avg_temp"
]
categorical_features = ["Area", "Item"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

models = {
    "Extra Trees Regressor": ExtraTreesRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "AdaBoost Regressor": AdaBoostRegressor(
        n_estimators=50, random_state=42
    ),
    "Elastic Net": ElasticNet(
        alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=42
    ),
    "HistGradientBoosting Regressor": HistGradientBoostingRegressor(
        max_iter=100, learning_rate=0.1, random_state=42
    ),
    "SGD Regressor": SGDRegressor(
        max_iter=2000, tol=1e-3, early_stopping=True, random_state=42
    )
}

for name in models:
    print("-", name)

## 7. Run 10-Fold Cross-Validation

Each model is evaluated on the same ten folds. The main comparison metric is RMSE; **lower RMSE is better**. The notebook reports mean RMSE and standard deviation across folds.

In [ ]:
K = 10
cv = KFold(n_splits=K, shuffle=True, random_state=42)

results = []
fold_scores = {}
start_all = time.time()

for name, estimator in models.items():
    print(f"Running: {name}")
    pipe = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", clone(estimator))
    ])
    start = time.time()
    scores = cross_validate(
        pipe, X, y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
        return_train_score=False,
        error_score="raise"
    )
    rmse_values = -scores["test_score"]
    elapsed = time.time() - start
    fold_scores[name] = rmse_values
    results.append({
        "Algorithm": name,
        "Mean RMSE": float(np.mean(rmse_values)),
        "Std RMSE": float(np.std(rmse_values, ddof=1)),
        "Best Fold RMSE": float(np.min(rmse_values)),
        "Worst Fold RMSE": float(np.max(rmse_values)),
        "Time (seconds)": float(elapsed)
    })
    print(f"  Mean RMSE: {np.mean(rmse_values):,.2f}")
    print(f"  Time: {elapsed:.1f} seconds")

print(f"\nTotal CV time: {(time.time() - start_all)/60:.1f} minutes")

In [ ]:
results_df = pd.DataFrame(results).sort_values("Mean RMSE").reset_index(drop=True)
results_df.insert(0, "Rank", np.arange(1, len(results_df) + 1))
display(results_df.style.format({
    "Mean RMSE": "{:,.2f}",
    "Std RMSE": "{:,.2f}",
    "Best Fold RMSE": "{:,.2f}",
    "Worst Fold RMSE": "{:,.2f}",
    "Time (seconds)": "{:.1f}"
}))
results_df.to_csv("crop_yield_model_comparison.csv", index=False)
print("Saved: crop_yield_model_comparison.csv")

## 8. Visualize and save the model comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(results_df["Algorithm"], results_df["Mean RMSE"], xerr=results_df["Std RMSE"])
plt.xlabel("RMSE (hg/ha yield units)")
plt.ylabel("Regression algorithm")
plt.title("10-Fold Cross-Validation: Mean RMSE (lower is better)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("crop_yield_rmse_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
for name, values in fold_scores.items():
    plt.plot(range(1, K + 1), values, marker="o", label=name)
plt.xlabel("Fold")
plt.ylabel("RMSE (hg/ha yield units)")
plt.title("RMSE for Each Fold")
plt.xticks(range(1, K + 1))
plt.legend()
plt.tight_layout()
plt.savefig("crop_yield_fold_rmse.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Train the best model on the full dataset and save it

Cross-validation helps choose the algorithm. After comparing scores, this cell fits the selected pipeline on all available cleaned rows and saves the trained model for the web app.

In [ ]:
best_name = results_df.loc[0, "Algorithm"]
best_estimator = clone(models[best_name])

final_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", best_estimator)
])
final_model.fit(X, y)

joblib.dump(final_model, "crop_yield_model.joblib")

metadata = {
    "best_algorithm": best_name,
    "target_column": TARGET,
    "target_unit": "hectograms per hectare (hg/ha)",
    "features": FEATURES,
    "areas": sorted(df["Area"].dropna().astype(str).unique().tolist()),
    "crops": sorted(df["Item"].dropna().astype(str).unique().tolist()),
    "rows_used": int(len(df)),
    "year_min": int(df["Year"].min()),
    "year_max": int(df["Year"].max()),
    "mean_rmse": float(results_df.loc[0, "Mean RMSE"]),
    "std_rmse": float(results_df.loc[0, "Std RMSE"])
}
with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Best algorithm:", best_name)
print("Mean 10-fold RMSE:", f"{metadata['mean_rmse']:,.2f} hg/ha")
print("Trained final model on all", len(df), "rows.")
print("Saved crop_yield_model.joblib and model_metadata.json")

## 10. Demonstrate a numerical prediction

This example uses a real row from the dataset to demonstrate the prediction flow. It is not a claim that every future field will produce the same yield.

In [ ]:
example_input = X.iloc[[0]].copy()
actual_value = float(y.iloc[0])
predicted_value = float(final_model.predict(example_input)[0])

print("Example input:")
display(example_input)
print(f"Actual yield:    {actual_value:,.2f} hg/ha")
print(f"Predicted yield: {predicted_value:,.2f} hg/ha")
print(f"Predicted yield: {predicted_value / 10000:,.4f} tonnes/ha")

## 11. Download the files needed for deployment and submission

Download the trained model, metadata, comparison table, and charts. Put `crop_yield_model.joblib`, `model_metadata.json`, `app.py`, and `requirements.txt` together in the same deployment folder.

In [ ]:
from google.colab import files
import zipfile

artifact_files = [
    "crop_yield_model.joblib",
    "model_metadata.json",
    "crop_yield_model_comparison.csv",
    "crop_yield_rmse_comparison.png",
    "crop_yield_fold_rmse.png"
]
with zipfile.ZipFile("crop_yield_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in artifact_files:
        if os.path.exists(filename):
            zf.write(filename)

print("Created crop_yield_results.zip")
files.download("crop_yield_results.zip")

## 12. What to include in the final report

After you run the notebook, copy the **actual** values from `results_df` into your report:

- Dataset name and Kaggle link
- Number of rows used after cleaning
- Input features and target column
- Brief explanation of the five algorithms
- 10-fold cross-validation method
- Comparison table with mean RMSE and standard deviation
- RMSE charts
- Best algorithm based on lowest mean RMSE
- Example actual vs predicted yield
- Deployment link once the web app is live

Do not write invented scores. Your final results depend on the dataset and the run in Google Colab.